## Install Packages

In [ ]:
!pip install --upgrade transformers
!pip install accelerate torch
!pip install -U bitsandbytes


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Get the Tokenizers for Gemma and Qwen

In [ ]:
from transformers import AutoTokenizer

models = {
    "qwen": "Qwen/Qwen2.5-1.5B-Instruct",
    "gemma4": "google/gemma-4-E2B-it",
}

tokenizers = {name: AutoTokenizer.from_pretrained(path) for name, path in models.items()}

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You are using a model of type gemma4 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
sentences = [
    "सगरमाथा संसारको सबैभन्दा अग्लो हिमाल हो।",
    "मलाई नेपाली खाना, विशेष गरी दाल-भात र गुन्द्रुक धेरै मन पर्छ।",
    "के तपाईं भोलि बिहानै बजार जान तयार हुनुहुन्छ?",
    "अहिलेको डिजिटल युगमा प्रविधिको महत्व दिनप्रतिदिन बढ्दै गएको छ।",
    "बाटो काट्दा सधैं जेब्रा क्रसिङको प्रयोग गर्नुपर्छ।",
    "आजको मौसम निकै राम्रो छ, त्यसैले हामी वनभोज जाने योजना बनाउँदैछौं।",
    "काठमाडौंको ट्राफिक जामले गर्दा धेरै मानिसहरु समयमा अफिस पुग्न सक्दैनन्।",
    "हामीले आफ्नो गाउँ र सहरलाई सधैं सफा राख्नुपर्छ।",
    "नेपाल एउटा बहुभाषिक र बहुसांस्कृतिक देश हो।",
    "विदेशी पर्यटकहरु नेपालको प्राकृतिक सुन्दरता हेर्न लालायित हुन्छन्।",
    "तिमीले आफ्नो गृहकार्य समयमै सक्यौ कि नाइँ?",
    "स्वास्थ्य नै सबैभन्दा ठूलो धन हो भन्ने कुरा हामीले बिर्सनु हुँदैन।",
    "गौतम बुद्धको जन्म लुम्बिनीमा भएको थियो, जसलाई शान्तिको प्रतीक मानिन्छ।",
    "किसानहरु खेतबारीमा काम गर्न व्यस्त छन् किनकि असारको महिना सुरु भयो।",
    "मिहिनेत नै सफलताको मुख्य कडी हो, त्यसैले निरन्तर प्रयास गरिरहनुपर्छ।"
]

## Fertility Test

In [ ]:
import numpy as np

def fertility_score(tokenizer, sentences):
    scores = []

    for sentence in sentences:
        tokens = tokenizer.encode(sentence)
        words = sentence.split()

        if len(words) > 0:
            scores.append(len(tokens) / len(words))

    return np.mean(scores)

for name, tokenizer in tokenizers.items():
    score = fertility_score(tokenizer, sentences)
    print(f"{name}: {score:.2f} tokens per word")

qwen: 6.25 tokens per word
gemma4: 2.30 tokens per word


## Devanagri Token Counts in the Model's Vocab

In [ ]:
def is_devanagari(text):
    """Checks if a string contains Devanagari characters."""
    # Devanagari Unicode range: \u0900 to \u097f
    return any('\u0900' <= char <= '\u097f' for char in text)

for name, tokenizer in tokenizers.items():
    vocab = tokenizer.get_vocab()

    # Filter for Devanagari tokens
    # We strip the " " (meta-space) character to get the clean text
    devanagari_tokens = [
        token for token in vocab.keys()
        if is_devanagari(token.replace(' ', ''))
    ]

    print(f"--- Model: {name} ---")
    print(f"Total Vocab Size: {len(vocab)}")
    print(f"Devanagari Token Count: {len(devanagari_tokens)}")

    # Print 5 sample tokens to see how they look
    print(f"Sample Tokens: {devanagari_tokens[:5]}")
    print("-" * 30)

--- Model: qwen ---
Total Vocab Size: 151665
Devanagari Token Count: 0
Sample Tokens: []
------------------------------
--- Model: gemma4 ---
Total Vocab Size: 262144
Devanagari Token Count: 13754
Sample Tokens: ['▁जीवनी', 'र्डर', 'फना', '▁झग', 'ण्यास']
------------------------------


## Load the Gemma4 model as 4bit Quantized and Qwen as Normal.
## Measure the Resource Usage

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def get_real_mem():
    """Flushes cache and returns actual used VRAM in GB."""
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / (1024**3)

# --- Start Measurement ---
loaded_models = {}
vram_log = {}

# Baseline
baseline = get_real_mem()

# Load Qwen
print("Loading Qwen...")
loaded_models["qwen"] = AutoModelForCausalLM.from_pretrained(
    models["qwen"],
    device_map="auto",
    torch_dtype=torch.bfloat16
)
mem_after_qwen = get_real_mem()
vram_log["qwen"] = mem_after_qwen - baseline

# Load Gemma 4
print("Loading Gemma 4...")
gemma_quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

loaded_models["gemma4"] = AutoModelForCausalLM.from_pretrained(
    models["gemma4"],
    quantization_config=gemma_quant_config,
    device_map="auto"
)

# FORCE GEMMA TO INITIALIZE
# We run a tiny mock input to force the weights from 'lazy' to 'active'
dummy_input = torch.tensor([[1]]).to("cuda")
with torch.no_grad():
    _ = loaded_models["gemma4"](dummy_input)

mem_after_gemma = get_real_mem()
vram_log["gemma4"] = mem_after_gemma - mem_after_qwen

# --- Final Report ---
print("\n" + "="*30)
print(f"Qwen 2.5 Usage:   {vram_log['qwen']:.2f} GB")
print(f"Gemma 4 Usage:    {vram_log['gemma4']:.2f} GB")
print("-" * 30)
print(f"TOTAL STACKED:    {get_real_mem():.2f} GB")
print("="*30)

Loading Qwen...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading Gemma 4...


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]


Qwen 2.5 Usage:   2.88 GB
Gemma 4 Usage:    6.29 GB
------------------------------
TOTAL STACKED:    9.17 GB


## Check the Amount of Token Models Generate for a Devanagri Word

In [ ]:

text_to_tokenize = "काठमाडौँ"
for tokenizer in tokenizers.values():
  tokens = tokenizer.tokenize(text_to_tokenize)
  token_ids = tokenizer.encode(text_to_tokenize, add_special_tokens=False)
  print(f"Input Text: {text_to_tokenize}")
  print(f"Token List: {tokens}")
  print(f"Token IDs:  {token_ids}")

Input Text: काठमाडौँ
Token List: ['à¤ķ', 'à¤¾à¤', 'ł', 'à¤®', 'à¤¾à¤', '¡', 'à¥', 'Į', 'à¤', 'ģ']
Token IDs:  [64704, 31411, 254, 87244, 31411, 94, 12619, 234, 5502, 223]
Input Text: काठमाडौँ
Token List: ['का', 'ठमाड', 'ौ', 'ँ']
Token IDs:  [2194, 152606, 237172, 237565]


## Check the 0 Shot Response to a Devanagri Prompt

In [ ]:
# List of prompts
prompts = [
    "नेपालको राजधानी के हो?",
    "सगरमाथाको बारेमा जानकारी दिनुहोस् र यसको महत्वबारे थप वाक्यहरू लेख्नुहोस्।",
    "नेपाली झण्डाको विशेषता के हो?",

]

def generate(model, tokenizer, text):
    # Wrap the prompt in a chat structure
    messages = [
        {"role": "user", "content": text},
    ]

    # This automatically adds the <start_of_turn> tags Gemma needs
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    # Slice the output to remove the input prompt tokens
    input_len = inputs.input_ids.shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

# Loop through each prompt for comparative analysis
for i, prompt in enumerate(prompts, 1):
    print(f"\n--- Testing Prompt {i} ---")
    print(f"Prompt: {prompt}")

    print(f"\nGemma Output:")
    print(generate(loaded_models["gemma4"], tokenizers["gemma4"], prompt))

    print(f"\nQwen Output:")
    print(generate(loaded_models["qwen"], tokenizers["qwen"], prompt))
    print("-" * 30)


--- Testing Prompt 1 ---
Prompt: नेपालको राजधानी के हो?

Gemma Output:
नेपालको राजधानी **काठमाडौं** हो।

Qwen Output:
संग्रहालय: सिंहपौर
------------------------------

--- Testing Prompt 2 ---
Prompt: सगरमाथाको बारेमा जानकारी दिनुहोस् र यसको महत्वबारे थप वाक्यहरू लेख्नुहोस्।

Gemma Output:
## सगरमाथा (Mount Everest) को बारेमा जानकारी

**सगरमाथा** विश्वको सबैभन्दा अग्लो शिखर हो, जुन हिमालय पर्वत श्रृंखलाको भाग हो र नेपाल तथा चीनको सीमामा अवस्थित छ। यो पृथ्वीको सतहबाट सबैभन्दा बढी उचाइमा रहेको स्थान हो जसलाई मापन गरिन्छ।

### मुख्य जानकारी:

* **अग्नि:** सगरमाथाको आधिकारिक नाम 'माउन्ट एवरेस्ट' हो।
* **उच्चता:** यसको उचाइ समुद्र सतहबाट करिब ८,८४८.८६ मिटर (२९,०३२.१७ फिट) छ।
* **स्थान:** यो नेपाल (नेपाल क्षेत्र) र चीन (तिब्बत स्वायत्त क्षेत्र) बीच विभाजित छ।
* **भौगोलिक स्थिति:** यो विश्वको सर्वोच्च शिखर हो र हिमाली क्षेत्रको प्रतिनिधित्व गर्दछ।
* **निर्माण प्रक्रिया:** यो भूगर्भीय प्लेटहरूको टक्करले बनेको हिमाल समूहको एक हिस्सा हो।
* **चुनौतीपूर्ण यात्रा:** सगरमाथा आरोहण अत्यन्त जोखिमपूर

In [ ]:
from transformers import PreTrainedTokenizerFast

# Load the tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained("Aananda-giri/NepaliBPE")

# Example usage
print(tokenizer.tokenize('राम ले भात खायो ।'))
# ['राम</w>', 'ले</w>', 'भात</w>', 'खायो</w>', '।</w>']

tokenizer.encode('राम ले भात खायो ।')
# [1621, 285, 14413, 27675, 251]



['राम</w>', 'ले</w>', 'भात</w>', 'खायो</w>', '।</w>']


[1621, 285, 14413, 27675, 251]

In [ ]:
print(tokenizers["gemma4"].tokenize('राम ले भात खायो ।'))
tokenizers["gemma4"].encode('राम ले भात खायो ।')

['राम', '▁ले', '▁भा', 'त', '▁ख', 'ाय', 'ो', '▁।']


[21779, 2892, 5852, 236844, 2216, 4546, 236850, 4156]